# Task A: State Estimation (5P)

Consider the omnidirectional robot from Figure 1 with a state vector of the form $\mathbf{x} = [x \ y \ \dot{x} \  \dot{y}]^T \in \mathbb{R}^{4}$, where x and y are the position of the robot and $\dot{x}$ and $\dot{y}$ are the velocities. In this task, you will implement a Kalman filter to estimate the robot's state. For this, IMU ($x, y$ acceleration) and GPS (latitude, longitude) data are synchronized and acquired at a frequency of $10$ Hz. Random noise has been added to the IMU data for visualization purposes. Please solve the following tasks:


		

<img src="media/omnidirectional_robot_fig.png" width="" align="" />

Figure 1: Omnidirectional robot with position x, y in plane\.

## Part 1 (1P)
Given the IMU acceleration input $\mathbf{u} = [\ddot{x} \ \ddot{y}]^T$ and the GPS position update $\mathbf{z} = [x \ y]^T$, provide the discrete state space model for the Kalman filter prediction (matrices $\mathbf{A}, \mathbf{B}$) and update (matrix $\mathbf{H}$) equations with an integration time step of $\Delta t = 0.1 s$. (Text answer)

## Part 2 (2P) 
Write the equations for the prediction and measurement update in the functions *kf_predict* and *kf_update*. (Python answer) 

In [ ]:
# ADD YOUR CHANGES IN THIS SECTION FOR TASK 2.
!pip install pymap3d
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm
from scipy import linalg
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import pymap3d as pm
import json

# Fix the seed for reproducibility of the result
np.random.seed(10)

# Definition of variables

# X_prev = Estimated state vector at previous time step
# P_prev = Estimated process covariance matrix of the previous state

# X_pred = Predicted state vector at current time step
# P_pred = Predicted process covariance matrix of the current state

# X_updated = Updated state vector at current time step
# P_updated = Updated process covariance matrix of the current state

# A = State transition matrix
# B = Input matrix
# U = Input vector
# Q = Process noise covariance matrix estimating Wn.

# Z = Sensor measurment vector
# H = Measurement transition matrix
# R = Sensor noise covariance estimating Vn

# K = Kalman gain


# PREDICTION MODEL
def kf_predict(X_prev, P_prev, A, B, U, Q):
    X_pred = None   # TODO: State Equation
    P_pred = None   # TODO: Process Uncertainity Equation
    return(X_pred,P_pred)

# MEASUREMENT UPDATE MODEL
def kf_update(X_pred,P_pred,Z,H,R):
    K = None            # TODO: Kalman Gain Equation
    X_updated = None    # TODO: State Update Equation
    P = None            # TODO: Process Covariance Update Equation
    return(K,X_updated,P)

In [ ]:
# NO CHANGES ARE NEEDED IN THIS SECTION!!!

# DATASET: https://doi.org/10.5281/zenodo.6557994
# Load GPS data from JSON file
json_file_path = 'data/cleaned_imu_gps_data.json'
with open(json_file_path, 'r') as file:
    gps_data = json.load(file)

# Get the first entry as the reference location
first_entry = gps_data.get("0")  # Assuming the first entry exists
lat0 = first_entry['lat_degrees']
lon0 = first_entry['lon_degrees']
h0 = 0.0  # Assuming sea level for altitude

# Lists to store ENU coordinates and IMU acceleration
east = []
north = []
accel_x = []
accel_y = []
delta_times = []
N = 600 # Number of samples to read (e.g. 35000)

# Extract timestamps and calculate time differences
timestamps = [int(gps_data[str(i)]['datetime']) for i in range(N) if str(i) in gps_data]
delta_times = np.diff(timestamps) / 1000.0  # Convert milliseconds to seconds
dt = np.average(delta_times)
print("Delta time dt = ", dt)

# Process the first N samples
for i in range(N):
    sample = gps_data.get(str(i))  # Retrieve data by string key
    if sample:
        lat = sample['lat_degrees']
        lon = sample['lon_degrees']
        h = 0.0  # Assuming sea level for altitude

        # Convert lat/lon to ENU coordinates
        e, n, u = pm.geodetic2enu(lat, lon, h, lat0, lon0, h0)
        east.append(e)
        north.append(n)

        # Read IMU acceleration data and add Gaussian noise
        noise_x = np.random.normal(0, 4)  # Mean 0, std dev 4
        noise_y = np.random.normal(0, 4)
        accel_x.append(sample.get('accel_x_ms2', 0) + noise_x)
        accel_y.append(sample.get('accel_y_ms2', 0) + noise_y)

# Normalize timestamps for color mapping
time_norm = np.array(timestamps) - min(timestamps)

# Plot East vs North with color scale based on timestamp
plt.figure(figsize=(8, 6))
sc = plt.scatter(east, north, c=time_norm, cmap='viridis', marker='o')
cbar = plt.colorbar(sc, label='Time (relative)')
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.title('2D Plot of GPS Data (ENU Coordinates)')
plt.grid()
plt.show()

# Plot IMU Acceleration
plt.figure(figsize=(8, 6))
plt.plot(range(N), accel_x, marker='o', linestyle='-', markersize=4, label='Accel X')
plt.plot(range(N), accel_y, marker='s', linestyle='-', markersize=4, label='Accel Y')
plt.xlabel('Sample Index')
plt.ylabel('Acceleration (m/s²)')
plt.title('IMU Linear Acceleration on X and Y Axes with Gaussian Noise')
plt.legend()
plt.grid()
plt.show()


## Part 3 (1P) 
Define the state space matrices $\mathbf{A}, \mathbf{B}, \mathbf{H}$ and the noise covariance matrices $\mathbf{Q}, \mathbf{R}$. (Python answer)

In [ ]:
# ADD YOUR CHANGES IN THIS SECTION FOR TASK 3

# PREDICTION
# Initialization of state vector and proces covariance matrix
X_prev = np.array([[0],[0],[0],[0]])
P_prev = np.array([[0.1,0,0,0],[0,0.1,0,0],[0,0,0.1,0],[0,0,0,0.1]])
X_pred = []
P_pred = []

A = None              # TODO: define the A matrix. Hint: The delta time is dt.
B = None              # TODO: define the B matrix. Hint: The delta time is dt.

# IMU acceleration input
U = np.c_[accel_x, accel_y]

# Noise of the position and velocities in the process model
std_dev_x  = 0.5       # TODO: Tune the standard deviation of the noise on the x axis. Hint: consider the ratio between std_dev_x and std_dev_x_gps.
std_dev_y  = 0.5       # TODO: Tune the standard deviation of the noise on the y axis. Hint: consider the ratio between std_dev_y and std_dev_y_gps.
std_dev_x_dot = 0.1
std_dev_y_dot = 0.1
Q = None               # TODO: define the Q matrix taking into account the standard deviation of the process noise. Hint: variance is squared std_dev.

# UPDATE
# GPS measurements
X_measurement = east
Y_measurement = north
Z = np.c_[X_measurement, Y_measurement]

# Noise of GPS measurements
std_dev_x_gps = 0.5
std_dev_y_gps = 0.5
R = None                # TODO: define the R matrix taking into account the standard deviation of the measurement noise. Hint: variance is squared std_dev.
H = None                # TODO: define the H matrix

## Part 4 (0.5P) 
Call the prediction and update functions in the iterative $for$-loop. (Python answer)

In [ ]:
# ADD YOUR CHANGES IN THIS SECTION FOR TASK 4

x_position_predicted = []
x_position_updated = []
x_velocity_predicted = []
x_velocity_updated =[]
y_position_predicted = []
y_position_updated = []
y_velocity_predicted = []
y_velocity_updated =[]
p_updated =[]

for i in tqdm(range(1,len(X_measurement)+1)):
    X_pred, P_pred = None, None                  # TODO: Call the prediction function. Hint: use the input U[i - 1:i].T

    # Append the predicted position and velocity values from the x and y axes
    if X_pred is not None:        
        x_position_predicted = np.append(x_position_predicted,X_pred[0:1])
        y_position_predicted = np.append(y_position_predicted,X_pred[1:2])
        x_velocity_predicted = np.append(x_velocity_predicted,X_pred[2:3])
        y_velocity_predicted = np.append(y_velocity_predicted,X_pred[3:4])

    K, X_updated, P_updated = None, None, None   # TODO: Call the update function. Hint: use the measurement Z[i - 1:i].T

    # Append the updated position and velocity values from the x and y axes
    if X_updated is not None:        
        x_position_updated = np.append(x_position_updated,X_updated[0:1])
        y_position_updated = np.append(y_position_updated,X_updated[1:2])
        x_velocity_updated = np.append(x_velocity_updated,X_updated[2:3])
        y_velocity_updated = np.append(y_velocity_updated,X_updated[3:4])

    # Append the updated covariance matrix
    if P_updated is not None:        
        p_updated = np.append(p_updated,P_updated)

    # Updated state becomes previous state
    X_prev = X_updated
    P_prev = P_updated

print("\n")
print("len(x_position_updated) =", len(x_position_updated))
print("len(y_position_updated) =", len(y_position_updated))

In [ ]:
# NO CHANGES ARE NEEDED IN THIS SECTION!!!

# PLOTS
# Calculate uncertainty of the position and velocity on the x-axis estimates for a 95% confidence interval
confidence_interval = 0.95;
z_score = -norm.ppf((1-confidence_interval) / 2.0)

if len(p_updated) != 0:
    p_updated = np.array(p_updated).reshape(len(X_measurement), 4, 4)

    upper_bound_position_x = x_position_updated  + (z_score*np.sqrt(p_updated[:, 0, 0]))
    lower_bound_position_x = x_position_updated  - (z_score*np.sqrt(p_updated[:, 0, 0]))
    upper_bound_position_y = y_position_updated  + (z_score*np.sqrt(p_updated[:, 1, 1]))
    lower_bound_position_y = y_position_updated  - (z_score*np.sqrt(p_updated[:, 1, 1]))

    upper_bound_velocity_x = x_velocity_updated + (z_score*np.sqrt(p_updated[:, 2, 2]))
    lower_bound_velocity_x = x_velocity_updated - (z_score*np.sqrt(p_updated[:, 2, 2]))
    upper_bound_velocity_y = y_velocity_updated + (z_score*np.sqrt(p_updated[:, 3, 3]))
    lower_bound_velocity_y = y_velocity_updated - (z_score*np.sqrt(p_updated[:, 3, 3]))

    # Plot of predicted and updated position values
    plt.subplot(2, 2, 1)
    x_axis_ticks = np.arange(1,len(X_measurement)+1,1)
    plt.plot(x_axis_ticks*dt, x_position_predicted,color="orange",label = "predicted (IMU)")
    plt.plot(x_axis_ticks*dt, X_measurement,label = "measured (GPS)")
    plt.plot(x_axis_ticks*dt, x_position_updated,color="green",label = "estimated", linestyle='dashed')
    plt.fill_between(x_axis_ticks*dt, upper_bound_position_x, lower_bound_position_x, alpha=0.15, color='grey', label = "95% confidence interval")

    plt.xlabel('time [s]')
    plt.ylabel('x-axis position [m]')
    plt.title('x-axis Position Tracking')
    plt.legend()

    plt.subplot(2, 2, 3)
    y_axis_ticks = np.arange(1,len(Y_measurement)+1,1)
    plt.plot(y_axis_ticks*dt, y_position_predicted,color="orange",label = "predicted (IMU)")
    plt.plot(y_axis_ticks*dt, Y_measurement,label = "measured (GPS)")
    plt.plot(y_axis_ticks*dt, y_position_updated,color="green",label = "estimated", linestyle='dashed')
    plt.fill_between(y_axis_ticks*dt, upper_bound_position_y, lower_bound_position_y, alpha=0.15, color='grey', label = "95% confidence interval")

    plt.xlabel('time [s]')
    plt.ylabel('y-axis position [m]')
    plt.title('y-axis Position Tracking')
    plt.legend()

    # Plot of predicted and updated velocity values
    plt.subplot(2, 2, 2)
    plt.plot(x_axis_ticks*dt, x_velocity_predicted,color="orange",label = "predicted (IMU)")
    plt.plot(x_axis_ticks*dt, x_velocity_updated,color="green", label = "estimated", linestyle='dashed')
    plt.fill_between(x_axis_ticks*dt, upper_bound_velocity_x, lower_bound_velocity_x, alpha=0.15, color='grey', label = "95% confidence interval")

    plt.xlabel('time [s]')
    plt.ylabel('x-axis velocity [m/s]')
    plt.title('x-axis Velocity Tracking')
    plt.legend()

    plt.subplot(2, 2, 4)
    plt.plot(y_axis_ticks*dt, y_velocity_predicted,color="orange",label = "predicted (IMU)")
    plt.plot(y_axis_ticks*dt, y_velocity_updated,color="green", label = "estimated", linestyle='dashed')
    plt.fill_between(y_axis_ticks*dt, upper_bound_velocity_y, lower_bound_velocity_y, alpha=0.15, color='grey', label = "95% confidence interval")

    plt.xlabel('time [s]')
    plt.ylabel('y-axis velocity [m/s]')
    plt.title('y-axis Velocity Tracking')
    plt.legend()

    plt.subplots_adjust(left=0.1,
                        bottom=0.1,
                        right=2,
                        top=1.5,
                        wspace=0.3,
                        hspace=0.3)
    plt.show()

    # RMSE Calculation
    rmse_innovation_x = np.sqrt(mean_squared_error(x_position_predicted, X_measurement))
    rmse_innovation_y = np.sqrt(mean_squared_error(y_position_predicted, Y_measurement))
    print("RMSE innovation x [m]: ", rmse_innovation_x)
    print("RMSE innovation y [m]: ", rmse_innovation_y)

    rmse_residual_x = np.sqrt(mean_squared_error(x_position_updated, X_measurement))
    rmse_residual_y = np.sqrt(mean_squared_error(y_position_updated, Y_measurement))
    print("RMSE residual x [m]: ", rmse_residual_x)
    print("RMSE residual y [m]: ", rmse_residual_y)

## Part 5 (0.5P) 
Vary the process noise standard deviation *std_dev_x* and *std_dev_y* to larger and smaller values compared to *std_dev_x_gps* and *std_dev_y_gps*. What do you observe in the position tracking plots? (Text answer)  

**ANSWER HERE**